# SME Capital Matching — Data Cleaning

**Notebook 2 of 3.** This notebook is both a working script and a manual: every code cell has a plain-language explanation above it, so a non-technical reader can follow exactly what is being done and why.

**What this notebook does.** Notebook 1 *measured* how messy the raw data is. This notebook *fixes* it. We take the raw export and turn it into one tidy, analysis-ready table by:

1. giving the long survey-question column headers short, friendly names (and keeping the full mapping visible as a **data dictionary**);
2. trimming stray spaces everywhere;
3. stripping the underscore "padding" off numeric answers and parsing money into clean Rand numbers;
4. capping impossible outliers (anything above R2 billion is treated as a typo);
5. turning revenue text-bands into an ordered 0–6 scale plus a representative Rand figure;
6. turning ownership-percentage bands into numbers;
7. pulling the BEE level number out of text like "Level 1";
8. standardising province spellings and tidying cities and emails;
9. removing duplicate applications.

**Where this notebook sits in the workflow.** Everything the project reads and writes lives in one place — the **`Datasets` folder**. This notebook is the middle link in a three-step chain:

| Step | File | Folder |
|---|---|---|
| **Input** — the raw, messy export | `capital_matching_applications.csv` (1,186 rows) | `Datasets` |
| ⬇ *this notebook cleans it* | | |
| **Output** — the tidy, analysis-ready table | `Capital_Matching_Cleaned_Data.xlsx` | `Datasets` |
| ⬇ *Notebook 3 picks that file straight up* | | |
| **Final result** — every SME scored and tiered | `Funding_Readiness_Segmentation.xlsx` | `Datasets` |

So: **we read one file from the `Datasets` folder, and we save one file back into the same `Datasets` folder.** Notebook 3 then opens that saved file and carries on. Nothing is written anywhere else.

> This recipe follows the reference script `clean_capital_matching.py`. The one difference: the reference stitched together several raw files, whereas here we start from a single 1,186-row export, so there is nothing to stack — we clean the one file directly.

## Step 1 — Find the `Datasets` folder and load the raw data

Two things happen here.

**First, we locate the `Datasets` folder.** The notebook does not assume you started Jupyter from any particular place — it simply looks in the folder it is running from, and then in the folders above it, until it finds one called `Datasets`. That folder is the project's single filing cabinet: **everything is read from it, and everything is saved back into it.** Setting the location once, here, means no other cell in the notebook ever has to worry about where files live.

**Second, we read the raw CSV in as text** (every column as a string). That matters: if we let the computer guess types, it would silently mangle the messy money fields. Reading as text keeps every original character intact so *we* decide how to interpret it.

In [1]:
import re
from pathlib import Path

import pandas as pd
import numpy as np


# first things first: where is the Datasets folder? start where we are and walk upwards until we spot it
def find_datasets_folder():
    """Look in this folder, then the one above it, and so on, until we find 'Datasets'."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "Datasets").is_dir():
            return folder / "Datasets"
    raise FileNotFoundError(
        "Could not find a folder named 'Datasets'. Please open this notebook from "
        "inside the project folder (the one that contains 'Datasets')."
    )


DATASETS = find_datasets_folder()          # the one place we read from and write to
RAW_FILE = DATASETS / "capital_matching_applications.csv"     # what we read in
CLEAN_FILE = DATASETS / "Capital_Matching_Cleaned_Data.xlsx"  # what we will save at the end

print(f"Datasets folder : {DATASETS}")
print(f"Reading from    : {RAW_FILE.name}")
print(f"Will save to    : {CLEAN_FILE.name}")

# okay, let's load the raw applications - everything as text so nothing gets mangled on the way in
raw = pd.read_csv(RAW_FILE, dtype=str)
print(f"\nLoaded {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head(3)

Datasets folder : C:\Users\IC Clearwater\OneDrive\Documents\GitHub\SME_Capital_Funding_Optimization\Datasets
Reading from    : capital_matching_applications.csv
Will save to    : Capital_Matching_Cleaned_Data.xlsx

Loaded 1,186 rows x 27 columns


,Created,Name:: Name & Surname,Company Name:,Company Registration Number:,Industry,Province,Location of Head Office (address),Funding Ask (Amount):,City/Town,Type of Funding,...,Number of Shareholders,What percentage of your business is owned by individuals aged 18 to 35? (Please select one),What percentage of your business is owned by women? (Please select one),What percentage of your business is under Black ownership?,BEE Level,Area,What percentage of your business is owned by individuals living with a disability? (Please select one),Contact person Name and Surname,Email address:,Contact number:
0,2025-08-15 19:19:12,Elmarie Daniels,Daniels Manufacturing,2011/627905/07,Education,Gauteng,"736 Biko Road, Kuruman",600000_________,Kuruman,Equity;Debt & Equity;Working Capital,...,1,0%,25–49%,100%,Level 1,Urban,0%,Elmarie Daniels,elmariedaniels@gmail.com,079 288 4800
1,2025-08-25 07:30:21,Suzette Fourie,Beacon Holdings,2013/679639/07,NaN,Gauteng,"1968 Church Road, Richards Bay",1190000________,Richards Bay,Working Capital,...,1,0%,0%,100%,Level 1,Township,0%,Suzette Fourie,suzettefourie@mweb.co.za,+27782300719
2,2025-08-14 18:10:50,Hendrik Van,VAN CIVILS,2019/949028/07,Transportation and Logistics,Western Cape,"76 Main Crescent, Centurion",3450000________,Centurion,Debt & Equity;Invoice Financing,...,1,100%,100%,100%,Level 3,Rural,0%,Hendrik Van,hendrikvan485@icloud.com,064 412 0771


## Step 2 — The data dictionary (short, tidy column names)

The raw headers are entire survey questions, several with trailing spaces (`Company Overview `, `Existing number of employees `). Long, inconsistent names are painful to work with, so we map each one to a short, code-friendly name. The full mapping below **is** the data dictionary — keep it as the reference for what every short name means.

In [2]:
# our data dictionary: raw survey question  ->  short tidy name
COLUMN_RENAME = {
    "Created": "created_at",
    "Name:: Name & Surname": "applicant_name",
    "Company Name:": "company_name",
    "Company Registration Number:": "company_reg_no",
    "Industry": "industry",
    "Province": "province",
    "Location of Head Office (address)": "head_office_address",
    "Funding Ask (Amount):": "funding_ask_raw",
    "City/Town": "city_town",
    "Type of Funding": "funding_type",
    "Funding Requirements (what is the funding required for).": "funding_purpose",
    "Annual Revenue Range 2023": "revenue_band_2023",
    "Annual Revenue Range 2024": "revenue_band_2024",
    "Actual Revenue 2025": "revenue_2025_raw",
    "Company Overview ": "company_overview",
    "Existing number of employees ": "employees_raw",
    "Increase of jobs anticipated through the capital access": "jobs_anticipated_raw",
    "Number of Shareholders": "shareholders_raw",
    "What percentage of your business is owned by individuals aged 18 to 35? (Please select one)": "youth_ownership_band",
    "What percentage of your business is owned by women? (Please select one)": "women_ownership_band",
    "What percentage of your business is under Black ownership? ": "black_ownership_band",
    "BEE Level": "bee_level_raw",
    "Area": "area",
    "What percentage of your business is owned by individuals living with a disability? (Please select one)": "disability_ownership_band",
    "Contact person Name and Surname": "contact_name",
    "Email address:": "email",
    "Contact number:": "contact_number",
}

df = raw.rename(columns=COLUMN_RENAME).copy()

# show the dictionary as a neat table so the reader can see every rename at a glance
data_dictionary = pd.DataFrame(
    [(orig, short) for orig, short in COLUMN_RENAME.items()],
    columns=["Original survey column", "Tidy name"],
)
display(data_dictionary)

,Original survey column,Tidy name
0,Created,created_at
1,Name:: Name & Surname,applicant_name
2,Company Name:,company_name
3,Company Registration Number:,company_reg_no
4,Industry,industry
5,Province,province
6,Location of Head Office (address),head_office_address
7,Funding Ask (Amount):,funding_ask_raw
8,City/Town,city_town
9,Type of Funding,funding_type


## Step 3 — The small "translator" helpers

Before we clean, we define a handful of tiny helper functions. Each one does a single, plainly-named job:

- **`strip_text`** — trims surrounding spaces and collapses double spaces; turns empty or `"nan"` text into a true blank.
- **`parse_money`** — turns a messy money answer like `250000_________` or `R 1,850,000` into the clean number `250000` / `1850000`.
- **`parse_percent_band`** — turns an ownership band like `50–74%` into a representative midpoint number (62). It treats the fancy en-dash `–` and the plain hyphen `-` as the same.
- **`parse_bee_level`** — pulls the number out of `"Level 1"` (in B-BBEE, **Level 1 is the best**).
- **`fix_province`** — maps spelling variants (`kzn`, `KwaZulu Natal`) onto one standard name.

We also set up the lookup tables for revenue bands (an ordered 0–6 scale and a Rand midpoint for each).

In [3]:
# a few tiny translators, each doing one clear job

def strip_text(x):
    """Trim spaces, squash double spaces; empty/'nan' becomes a real blank (NaN)."""
    if pd.isna(x):
        return np.nan
    s = re.sub(r"\s+", " ", str(x)).strip()
    return np.nan if s == "" or s.lower() == "nan" else s

def parse_money(x):
    """Turn a messy money answer into a plain number of Rand.
    The form padded numbers with underscores (e.g. '50000______'); people also
    typed 'R', commas and spaces. We keep only the digits and the decimal point."""
    if pd.isna(x):
        return np.nan
    s = re.sub(r"[^0-9.]", "", str(x))     # keep digits and dots only
    s = re.sub(r"\.(?=.*\.)", "", s)        # if several dots, keep just the last
    if s in ("", "."):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

# revenue text-bands -> an ordered scale (0 = smallest ... 6 = largest)
REVENUE_BAND_ORDER = {
    "0 to R500K": 0, "R501K to R1M": 1, "R1 000 001 to R5M": 2, "R5 000 001 to R10M": 3,
    "R10 000 001 to R20M": 4, "R20 000 001 to R50M": 5, "Above R50M": 6,
}
# ...and a rough Rand midpoint for each band (handy for sizing)
REVENUE_BAND_MIDPOINT = {
    "0 to R500K": 250_000, "R501K to R1M": 750_000, "R1 000 001 to R5M": 3_000_000,
    "R5 000 001 to R10M": 7_500_000, "R10 000 001 to R20M": 15_000_000,
    "R20 000 001 to R50M": 35_000_000, "Above R50M": 75_000_000,
}

# ownership percentage bands -> a representative midpoint percentage
PERCENT_BAND_MIDPOINT = {"0%": 0.0, "1–24%": 12.5, "25–49%": 37.0, "50–74%": 62.0, "75–99%": 87.0, "100%": 100.0}

def parse_percent_band(x):
    """Ownership band -> midpoint number. Handles en-dash '–' and hyphen '-' the same."""
    if pd.isna(x):
        return np.nan
    return PERCENT_BAND_MIDPOINT.get(str(x).strip().replace("-", "–"), np.nan)

def parse_bee_level(x):
    """Pull the level number out of 'Level 1' etc. (Level 1 is the BEST B-BBEE score)."""
    if pd.isna(x):
        return np.nan
    m = re.search(r"(\d+)", str(x))
    return int(m.group(1)) if m else np.nan

# province spelling variants -> one standard spelling
PROVINCE_FIX = {
    "kzn": "KwaZulu-Natal", "kwazulu natal": "KwaZulu-Natal", "kwazulu-natal": "KwaZulu-Natal",
    "gauteng": "Gauteng", "limpopo": "Limpopo", "western cape": "Western Cape",
    "eastern cape": "Eastern Cape", "northern cape": "Northern Cape", "north west": "North West",
    "free state": "Free State", "mpumalanga": "Mpumalanga",
}
def fix_province(x):
    if pd.isna(x):
        return np.nan
    return PROVINCE_FIX.get(str(x).strip().lower(), str(x).strip())

print("Helper functions and lookup tables are ready.")

Helper functions and lookup tables are ready.


## Step 4 — Clean every field

Now we run the translators across the data, one group of fields at a time. In order, we:

- trim whitespace on every text column;
- parse the timestamp;
- standardise province spellings and title-case cities so `johannesburg` and `Johannesburg ` become one place;
- parse the money fields, then **cap absurd outliers**: anything above R2 billion is almost certainly a typo (someone held the `0` key), so we blank it out;
- convert both revenue-band columns into the ordered 0–6 scale plus a Rand midpoint;
- convert employee, job and shareholder counts to numbers;
- convert the four ownership-percentage bands to midpoint numbers;
- pull out the BEE level number;
- lower-case emails so `Jo@X.com` and `jo@x.com` are recognised as the same address later.

In [4]:
# right, let's actually clean everything now

# 4a. trim whitespace on every column.
# every column was read in as text, so we simply run the trimmer across all of them.
# (we deliberately do NOT test the column "type" first: different versions of pandas label
#  text columns differently, and such a test would silently skip this whole step on some
#  machines - leaving stray spaces behind and quietly changing the scores later on.)
for c in df.columns:
    df[c] = df[c].map(strip_text)

# 4b. parse the application timestamp into a real date/time
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

# 4c. tidy the categorical text fields
df["province"] = df["province"].map(fix_province)
df["industry"] = df["industry"].str.strip()
df["city_town"] = df["city_town"].str.title()      # 'cape town ' and 'CAPE TOWN' -> 'Cape Town'

# 4d. parse the money fields (strip the underscore padding, 'R', commas...)
df["funding_ask_zar"] = df["funding_ask_raw"].map(parse_money)
df["revenue_2025_zar"] = df["revenue_2025_raw"].map(parse_money)

# 4e. cap impossible outliers: above R2 billion is treated as a data-entry error -> blank
OUTLIER_CAP = 2_000_000_000
outliers_removed = 0
for col in ["funding_ask_zar", "revenue_2025_zar"]:
    n = int((df[col] > OUTLIER_CAP).sum())
    outliers_removed += n
    df.loc[df[col] > OUTLIER_CAP, col] = np.nan

# 4f. revenue text-bands -> ordered scale + Rand midpoint, for both years
for yr in ["2023", "2024"]:
    band = f"revenue_band_{yr}"
    df[f"revenue_scale_{yr}"] = df[band].map(REVENUE_BAND_ORDER)
    df[f"revenue_mid_{yr}"] = df[band].map(REVENUE_BAND_MIDPOINT)

# 4g. numeric counts
df["employees"] = pd.to_numeric(df["employees_raw"], errors="coerce")
df["jobs_anticipated"] = pd.to_numeric(df["jobs_anticipated_raw"], errors="coerce")
df["shareholders"] = pd.to_numeric(df["shareholders_raw"], errors="coerce")

# 4h. ownership percentage bands -> midpoint numbers
df["youth_ownership_pct"] = df["youth_ownership_band"].map(parse_percent_band)
df["women_ownership_pct"] = df["women_ownership_band"].map(parse_percent_band)
df["black_ownership_pct"] = df["black_ownership_band"].map(parse_percent_band)
df["disability_ownership_pct"] = df["disability_ownership_band"].map(parse_percent_band)

# 4i. BEE level number
df["bee_level"] = df["bee_level_raw"].map(parse_bee_level)

# 4j. lower-case emails for reliable de-duplication later
df["email"] = df["email"].str.lower()

# a safety net: prove the trim actually happened. if this ever prints anything other than 0,
# the cleaning has not worked and the numbers further down cannot be trusted.
stray_spaces = 0
for c in df.columns:
    stray_spaces += int(df[c].map(lambda v: isinstance(v, str) and v != v.strip()).sum())

print(f"Cleaning done. Impossible (> R2bn) money values blanked out: {outliers_removed}")
print(f"Text values still carrying stray spaces (must be 0): {stray_spaces}")
print("\nA few cleaned columns to eyeball:")
df[["company_name", "province", "city_town", "funding_ask_zar", "revenue_2025_zar",
    "revenue_scale_2024", "bee_level", "black_ownership_pct"]].head(6)

Cleaning done. Impossible (> R2bn) money values blanked out: 12
Text values still carrying stray spaces (must be 0): 0

A few cleaned columns to eyeball:


,company_name,province,city_town,funding_ask_zar,revenue_2025_zar,revenue_scale_2024,bee_level,black_ownership_pct
0,Daniels Manufacturing,Gauteng,Kuruman,600000.0,603752.0,1,1,100.0
1,Beacon Holdings,Gauteng,Richards Bay,1190000.0,298746.0,0,1,100.0
2,VAN CIVILS,Western Cape,Centurion,3450000.0,2297089.0,2,3,100.0
3,Clear Water Agri,Gauteng,Worcester,350000.0,70549.0,0,1,100.0
4,Greenfield Foods CC,Gauteng,Kuruman,3300000.0,1098973.0,2,1,100.0
5,Sekgobela Group (Pty) Ltd,Western Cape,Ladysmith,30000.0,0.0,0,1,100.0


## Step 5 — Remove duplicate applications

The raw export contains repeats. We remove them in two passes, gentlest first:

1. **Exact duplicates** — rows that are identical in every single field. The first copy is kept, the rest dropped.
2. **Near-duplicates** — the same **company + email + submission time**. This catches the same person's application appearing twice with a tiny difference somewhere. Again we keep the first.

We print the before/after counts so the shrinkage is fully transparent.

In [5]:
# let's take out the repeated applications, keeping the first copy each time
rows_in = len(df)

# pass 1: fully identical rows
df = df.drop_duplicates()
after_exact = len(df)

# pass 2: same company + email + submission time (a re-submission of the same application)
df = df.drop_duplicates(subset=["company_name", "email", "created_at"], keep="first").reset_index(drop=True)
rows_out = len(df)

print(f"Rows in                     : {rows_in:,}")
print(f"After removing exact copies : {after_exact:,}  (-{rows_in - after_exact})")
print(f"After removing near-copies  : {rows_out:,}  (-{after_exact - rows_out})")
print(f"Unique businesses remaining : {rows_out:,}")

Rows in                     : 1,186
After removing exact copies : 1,118  (-68)
After removing near-copies  : 1,116  (-2)
Unique businesses remaining : 1,116


## Step 6 — Before / after summary

A quick side-by-side of what changed, so the impact of the clean is easy to explain in one glance: how many rows we started and ended with, and how many messy money values were rescued into real numbers.

In [6]:
# a plain before-and-after scoreboard
def clean_number_count(series):
    return int(series.map(lambda x: bool(re.fullmatch(r"\d+(\.\d+)?", str(x).strip())) if pd.notna(x) else False).sum())

summary = pd.DataFrame(
    [
        ("Rows (applications)", f"{rows_in:,}", f"{rows_out:,}"),
        ("Funding ask: usable numbers", f"{clean_number_count(raw['Funding Ask (Amount):']):,}", f"{int(df['funding_ask_zar'].notna().sum()):,}"),
        ("2025 revenue: usable numbers", f"{clean_number_count(raw['Actual Revenue 2025']):,}", f"{int(df['revenue_2025_zar'].notna().sum()):,}"),
        ("Distinct province spellings", f"{raw['Province'].str.strip().nunique():,}", f"{df['province'].nunique():,}"),
        ("BEE level as a number", "0", f"{int(df['bee_level'].notna().sum()):,}"),
    ],
    columns=["Measure", "Before (raw)", "After (clean)"],
)
display(summary)
print("The underscore-padded money fields are now real numbers; provinces are standardised; "
      "duplicates are gone. The data is ready to score.")

,Measure,Before (raw),After (clean)
0,Rows (applications),"1,186","1,116"
1,Funding ask: usable numbers,10,"1,109"
2,2025 revenue: usable numbers,0,"1,112"
3,Distinct province spellings,9,9
4,BEE level as a number,0,"1,116"


The underscore-padded money fields are now real numbers; provinces are standardised; duplicates are gone. The data is ready to score.


## Step 7 — Save the clean dataset back into the `Datasets` folder

Finally we save the cleaned table as a **presentable Excel file** — `Capital_Matching_Cleaned_Data.xlsx` — back into the same `Datasets` folder we read the raw CSV from.

It is deliberately simple: **one sheet, one table** of the cleaned data, with a styled header row and a frozen top row so it is easy to scroll. No summaries, no extra sheets — just the clean dataset.

**This file is the handover.** Notebook 3 opens exactly this file, out of exactly this folder, and uses it to score every business. Nothing else passes between the two notebooks, so if this file saves correctly, the next step will run.

In [7]:
# save the clean data as a single, tidy Excel table, straight back into the Datasets folder
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# strip the handful of invisible / non-standard characters Excel refuses to store
# (stray control codes and Unicode "non-characters" that hide in free-text answers).
# Normal text and emoji are kept.
def clean_cell(v):
    if not isinstance(v, str):
        return v
    return "".join(
        ch for ch in v
        if ch in "\t\n\r" or (
            ord(ch) >= 0x20 and ord(ch) != 0x7f
            and not (0x80 <= ord(ch) <= 0x9f)
            and not (0xFDD0 <= ord(ch) <= 0xFDEF)
            and (ord(ch) & 0xFFFF) not in (0xFFFE, 0xFFFF)
        )
    )

NAVY="1F3864"; FONT="Arial"
thin=Side(style="thin",color="D9D9D9"); BORDER=Border(left=thin,right=thin,top=thin,bottom=thin)

wb=Workbook(); ws=wb.active; ws.title="Clean Data"; ws.sheet_view.showGridLines=False

# header row
for j,col in enumerate(df.columns,1):
    cell=ws.cell(row=1,column=j,value=str(col))
    cell.font=Font(name=FONT,bold=True,color="FFFFFF",size=10)
    cell.fill=PatternFill("solid",fgColor=NAVY)
    cell.alignment=Alignment(horizontal="center",vertical="center",wrap_text=True)
    cell.border=BORDER

# data rows
money_cols={"funding_ask_zar","revenue_2025_zar","revenue_mid_2023","revenue_mid_2024"}
for i,(_,r) in enumerate(df.iterrows(),2):
    for j,col in enumerate(df.columns,1):
        v=r[col]
        if pd.isna(v): v=None
        elif isinstance(v,(np.integer,)): v=int(v)
        elif isinstance(v,(np.floating,)): v=float(v)
        cell=ws.cell(row=i,column=j,value=clean_cell(v))
        cell.font=Font(name=FONT,size=9); cell.border=BORDER; cell.alignment=Alignment(vertical="center")
        if col in money_cols: cell.number_format='#,##0;(#,##0);-'
ws.freeze_panes="A2"

# sensible column widths
for c in ws.columns:
    letter=get_column_letter(c[0].column)
    length=max((len(str(cell.value)) if cell.value is not None else 0) for cell in c)
    ws.column_dimensions[letter].width=min(max(length+2,10),40)

# CLEAN_FILE was set right at the top - it points into the Datasets folder
wb.save(CLEAN_FILE)
print(f"Saved the clean data table -> {CLEAN_FILE}")
print(f"({df.shape[0]:,} rows x {df.shape[1]} columns)")
print("\nNotebook 3 (Funding_Readiness_Segmentation) now reads this exact file. Over to it.")

Saved the clean data table -> C:\Users\IC Clearwater\OneDrive\Documents\GitHub\SME_Capital_Funding_Optimization\Datasets\Capital_Matching_Cleaned_Data.xlsx
(1,116 rows x 41 columns)

Notebook 3 (Funding_Readiness_Segmentation) now reads this exact file. Over to it.
